# Batched Attention-Mask Ablation vs Physical Chunk Deletion — Benchmark

Runs both ablation methods on the same needle-in-a-haystack prompts and compares:

| | Physical deletion (`chunk_deletion_baseline.py`) | Batched mask (`attention_mask_ablation.py`) |
|---|---|---|
| RoPE positions | **Shifted** for every token after deleted chunk | **Preserved** — `input_ids` never changes |
| Forward passes | `m + 1` (one per chunk) | `ceil(m / B) + 1` (one per batch) |
| Input length | Shrinks by `chunk_size` per ablation | **Constant** across all runs |
| VRAM per pass | Smaller (shorter sequence) | Larger (full length × batch size `B`) |

Setting `--ablation_batch_size 0` batches all `m` chunks into a single `model.generate()` call.

> **Before running:** *Runtime → Change runtime type → T4 GPU*

In [ ]:
# ── Check GPU ─────────────────────────────────────────────────────────────
import subprocess, torch

out = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,memory.free', '--format=csv,noheader'],
    capture_output=True, text=True
)
gpu_info = out.stdout.strip()
print('GPU    :', gpu_info or 'NOT FOUND — set Runtime to T4 GPU')
print('PyTorch:', torch.__version__)
print('CUDA   :', torch.cuda.is_available())
if torch.cuda.is_available():
    free, total = torch.cuda.mem_get_info(0)
    print(f'VRAM   : {total/1e9:.1f} GB total, {free/1e9:.1f} GB free')

if not torch.cuda.is_available():
    raise RuntimeError('No GPU found. Go to Runtime > Change runtime type > T4 GPU.')

In [ ]:
# ── Install dependencies ──────────────────────────────────────────────────
!pip install -q transformers>=4.40 sentence-transformers accelerate numpy matplotlib
print('Dependencies ready.')

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────
# Edit REPO_URL before running.
REPO_URL        = 'https://github.com/YOUR_USERNAME/context-utilization'  # <-- set this

MODEL_ID        = 'meta-llama/Llama-3.2-1B-Instruct'
DTYPE           = 'float16'        # faster than bfloat16 on T4
CHUNK_SIZE      = 128              # tokens per chunk
MAX_NEW_TOKENS  = 64
PROMPT_LENGTHS  = ['512', '1k', '2k']   # ~4, 8, 16 chunks each
NUM_EXAMPLES    = 1

# Batched mask settings
# 0 = all chunks in one generate() call (fastest wall time, highest VRAM)
# Set to e.g. 4 or 8 if you OOM
ABLATION_BATCH_SIZE = 0

OUTPUT_DIR_DEL  = '/content/context-utilization/results_deletion'
OUTPUT_DIR_MASK = '/content/context-utilization/results_mask'
PROMPTS_FILE    = 'test_prompts.jsonl'

print('Config ready.')

In [ ]:
# ── Clone repo (idempotent) ────────────────────────────────────────────────
import os, sys

REPO_PATH = '/content/context-utilization'

if not os.path.exists(REPO_PATH):
    result = subprocess.run(
        ['git', 'clone', REPO_URL, REPO_PATH],
        capture_output=True, text=True
    )
    print(result.stderr or result.stdout)
    if result.returncode != 0:
        raise RuntimeError('git clone failed:\n' + result.stderr)
else:
    print('Repo already present — pulling latest...')
    subprocess.run(['git', '-C', REPO_PATH, 'pull', '--ff-only'],
                   capture_output=True, text=True)

os.chdir(REPO_PATH)
sys.path.insert(0, REPO_PATH)
print('CWD:', os.getcwd())
print('Scripts:', sorted(f for f in os.listdir('.') if f.endswith('.py')))

In [ ]:
# ── HuggingFace login ─────────────────────────────────────────────────────
# Accept Meta's license at https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct
# then create a read-only token at https://huggingface.co/settings/tokens
from huggingface_hub import notebook_login
notebook_login()

In [ ]:
# ── Generate needle-in-a-haystack prompts ─────────────────────────────────
import json

cmd = ([sys.executable, 'generate_ruler_prompts.py',
        '--lengths']  + PROMPT_LENGTHS +
       ['--positions', 'middle',
        '--num_examples', str(NUM_EXAMPLES),
        '--output_file', PROMPTS_FILE,
        '--include_metadata'])

result = subprocess.run(cmd, capture_output=True, text=True)
print(result.stderr)
if result.returncode != 0:
    print(result.stdout)
    raise RuntimeError('Prompt generation failed')

with open(PROMPTS_FILE) as f:
    prompts = [json.loads(l) for l in f if l.strip()]

print(f'Generated {len(prompts)} prompts:')
for p in prompts:
    tl    = (p.get('metadata') or {}).get('target_tokens', '?')
    approx = len(p['prompt']) // 4
    ref   = p['reference']
    print(f'  target={tl:<6}  approx={approx:<6} tokens  needle={ref}')

In [ ]:
# ── Run 1 of 2: Physical chunk deletion baseline ───────────────────────────
# Estimated runtime on T4:
#   512  tokens × 128 chunk =  5 runs  ~  1-2 min
#   1024 tokens × 128 chunk =  9 runs  ~  3-4 min
#   2048 tokens × 128 chunk = 17 runs  ~  6-8 min
#   Total: ~12-15 min

os.makedirs(OUTPUT_DIR_DEL, exist_ok=True)

cmd = [sys.executable, 'chunk_deletion_baseline.py',
       '--model',          MODEL_ID,
       '--input_file',     PROMPTS_FILE,
       '--chunk_size',     str(CHUNK_SIZE),
       '--max_new_tokens', str(MAX_NEW_TOKENS),
       '--dtype',          DTYPE,
       '--output_dir',     OUTPUT_DIR_DEL]

print('Command:', ' '.join(cmd))
print('Running physical deletion...')
# stderr=subprocess.STDOUT merges stderr into stdout so all progress and
# error messages appear inline in the cell output.
result = subprocess.run(cmd, check=False, stderr=subprocess.STDOUT)
if result.returncode != 0:
    raise RuntimeError(
        f'chunk_deletion_baseline.py exited with code {result.returncode}.\n'
        'Scroll up in this cell to see the script output and error message.'
    )
print('\nPhysical deletion complete.')


In [ ]:
# ── Run 2 of 2: Batched attention-mask ablation ───────────────────────────
# Requires the updated attention_mask_ablation.py (with --ablation_batch_size)
# to be pushed to your GitHub repo and cloned above.
# If you see exit code 2 here, run: git add attention_mask_ablation.py &&
# git commit -m 'batched mask' && git push — then re-run the clone cell.
#
# Estimated runtime on T4 (ABLATION_BATCH_SIZE=0, all chunks at once):
#   512  tokens → 1 batched generation  ~  1-2 min
#   1024 tokens → 1 batched generation  ~  2-3 min
#   2048 tokens → 1 batched generation  ~  3-5 min
#   Total: ~6-10 min  (vs ~12-15 min for physical deletion)

# Pre-flight: confirm the script supports --ablation_batch_size
check = subprocess.run(
    [sys.executable, 'attention_mask_ablation.py', '--help'],
    capture_output=True, text=True
)
if 'ablation_batch_size' not in check.stdout:
    raise RuntimeError(
        'The cloned attention_mask_ablation.py does not have --ablation_batch_size.\n'
        'Push the updated script to GitHub, then re-run the clone cell above.'
    )

os.makedirs(OUTPUT_DIR_MASK, exist_ok=True)

cmd = [sys.executable, 'attention_mask_ablation.py',
       '--model',                MODEL_ID,
       '--input_file',           PROMPTS_FILE,
       '--chunk_size',           str(CHUNK_SIZE),
       '--max_new_tokens',       str(MAX_NEW_TOKENS),
       '--ablation_batch_size',  str(ABLATION_BATCH_SIZE),
       '--dtype',                DTYPE,
       '--output_dir',           OUTPUT_DIR_MASK]

print('Command:', ' '.join(cmd))
print('Running batched mask ablation...')
result = subprocess.run(cmd, check=False, stderr=subprocess.STDOUT)
if result.returncode != 0:
    raise RuntimeError(
        f'attention_mask_ablation.py exited with code {result.returncode}.\n'
        'Scroll up in this cell to see the script output and error message.'
    )
print('\nBatched mask ablation complete.')


In [ ]:
# ── Load results ──────────────────────────────────────────────────────────
import numpy as np

with open(os.path.join(OUTPUT_DIR_DEL,  'profiling_summary.json')) as f:
    prof_del  = json.load(f)
with open(os.path.join(OUTPUT_DIR_MASK, 'profiling_summary.json')) as f:
    prof_mask = json.load(f)

with open(os.path.join(OUTPUT_DIR_DEL,  'chunk_deletion_results.json')) as f:
    full_del  = json.load(f)
with open(os.path.join(OUTPUT_DIR_MASK, 'mask_ablation_results.json')) as f:
    full_mask = json.load(f)

# ── Comparison table ──────────────────────────────────────────────────────
print(f'Model     : {prof_del["model"]}')
print(f'Chunk size: {prof_del["chunk_size"]} tokens')
print(f'Batch size: {prof_mask["examples"][0]["ablation_batch_size"]} (0 = all chunks)')
print()

hdr = (f'{"Tokens":>8}  {"Chunks":>7}  '
       f'{"Del time(s)":>12}  {"Mask time(s)":>13}  '
       f'{"Speedup":>8}  {"Del passes":>11}  {"Mask batches":>13}  '
       f'{"Del VRAM":>10}  {"Mask VRAM":>10}')
print(hdr)
print('-' * len(hdr))

for ex_d, ex_m in zip(prof_del['examples'], prof_mask['examples']):
    pt      = ex_d['prompt_tokens']
    nc      = ex_d['num_chunks']
    td      = ex_d['total_wall_time_s']
    tm      = ex_m['total_wall_time_s']
    speedup = td / tm if tm > 0 else float('inf')
    passes_d = ex_d['total_ablation_runs']
    batches_m = ex_m['total_ablation_batches']
    vd      = ex_d['max_peak_vram_mb']
    vm      = ex_m['max_peak_vram_mb']
    print(
        f'{pt:>8}  {nc:>7}  '
        f'{td:>12.2f}  {tm:>13.2f}  '
        f'{speedup:>7.2f}x  {passes_d:>11}  {batches_m:>13}  '
        f'{vd:>10.0f}  {vm:>10.0f}'
    )

In [ ]:
# ── Plot 1: Wall time comparison (grouped bars per prompt length) ──────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

examples_d = prof_del['examples']
examples_m = prof_mask['examples']
n = len(examples_d)

labels   = [str(ex['prompt_tokens']) + ' tok' for ex in examples_d]
t_del    = [ex['total_wall_time_s'] for ex in examples_d]
t_mask   = [ex['total_wall_time_s'] for ex in examples_m]
speedups = [d / m for d, m in zip(t_del, t_mask)]

x = np.arange(n)
w = 0.35

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))

# Grouped bar: total wall time
b1 = ax1.bar(x - w/2, t_del,  width=w, color='coral',      label='Physical deletion')
b2 = ax1.bar(x + w/2, t_mask, width=w, color='steelblue',  label='Batched mask')
ax1.bar_label(b1, fmt='%.1fs', padding=3, fontsize=9)
ax1.bar_label(b2, fmt='%.1fs', padding=3, fontsize=9)
ax1.set_xticks(x)
ax1.set_xticklabels(labels)
ax1.set_xlabel('Prompt length')
ax1.set_ylabel('Total wall time (s)')
ax1.set_title('Total Wall Time')
ax1.legend()

# Speedup factor
bars = ax2.bar(labels, speedups, color='mediumseagreen', edgecolor='white')
ax2.bar_label(bars, fmt='%.2fx', padding=3, fontsize=10)
ax2.axhline(1.0, color='grey', linestyle='--', linewidth=0.8, label='no speedup')
ax2.set_xlabel('Prompt length')
ax2.set_ylabel('Speedup (deletion time / mask time)')
ax2.set_title('Speedup from Batching')
ax2.legend()

fig.suptitle('Batched Mask vs Physical Deletion — Wall Time', fontsize=13)
plt.tight_layout()
plt.savefig('comparison_wall_time.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_wall_time.png')

In [ ]:
# ── Plot 2: Forward passes vs ablation batches ─────────────────────────────
passes_del   = [ex['total_ablation_runs']    for ex in examples_d]
batches_mask = [ex['total_ablation_batches'] for ex in examples_m]

x = np.arange(n)
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, passes_del,   width=w, color='coral',     label='Physical deletion (passes)')
b2 = ax.bar(x + w/2, batches_mask, width=w, color='steelblue', label='Batched mask (generate calls)')
ax.bar_label(b1, padding=3, fontsize=10)
ax.bar_label(b2, padding=3, fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Prompt length')
ax.set_ylabel('Count')
ax.set_title('Number of model.generate() calls\n(Physical deletion: m+1 passes  |  Batched mask: ceil(m/B)+1 calls)')
ax.legend()
plt.tight_layout()
plt.savefig('comparison_forward_passes.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_forward_passes.png')

In [ ]:
# ── Plot 3: Peak VRAM comparison ──────────────────────────────────────────
vram_del  = [ex['max_peak_vram_mb'] for ex in examples_d]
vram_mask = [ex['max_peak_vram_mb'] for ex in examples_m]

x = np.arange(n)
w = 0.35

fig, ax = plt.subplots(figsize=(8, 5))
b1 = ax.bar(x - w/2, vram_del,  width=w, color='coral',     label='Physical deletion')
b2 = ax.bar(x + w/2, vram_mask, width=w, color='steelblue', label='Batched mask')
ax.bar_label(b1, fmt='%.0f MB', padding=3, fontsize=9)
ax.bar_label(b2, fmt='%.0f MB', padding=3, fontsize=9)
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_xlabel('Prompt length')
ax.set_ylabel('Peak VRAM (MB)')
ax.set_title('Peak VRAM per Run\n'
             'Mask VRAM scales with batch size B; deletion VRAM is lower per pass\n'
             'but paid m+1 times')
ax.legend()
plt.tight_layout()
plt.savefig('comparison_vram.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_vram.png')

In [ ]:
# ── Plot 4: Per-chunk influence score equivalence ─────────────────────────
# Scatter plot: deletion influence (x) vs mask influence (y).
# Points near the diagonal y=x mean the two methods agree.
# Systematic divergence reveals the RoPE-shift artifact in physical deletion.

import matplotlib.cm as cm

n = len(full_del)
fig, axes = plt.subplots(1, n, figsize=(5.5 * n, 5))
if n == 1:
    axes = [axes]

for ax, res_d, res_m, ex_d in zip(axes, full_del, full_mask, examples_d):
    scores_d = [c['influence_score'] for c in res_d['chunk_influences']]
    scores_m = [c['influence_score'] for c in res_m['chunk_influences']]
    chunk_idx = list(range(len(scores_d)))

    scatter = ax.scatter(
        scores_d, scores_m,
        c=chunk_idx, cmap='plasma', s=60, edgecolors='white', linewidths=0.5
    )

    # y = x reference line
    lo = min(min(scores_d), min(scores_m))
    hi = max(max(scores_d), max(scores_m))
    ax.plot([lo, hi], [lo, hi], 'k--', linewidth=0.9, label='y = x (perfect agreement)')

    corr = np.corrcoef(scores_d, scores_m)[0, 1]
    pt   = ex_d['prompt_tokens']
    ax.set_title(f'{pt} tokens   Pearson r = {corr:.4f}', fontsize=10)
    ax.set_xlabel('Influence — physical deletion')
    ax.set_ylabel('Influence — batched mask')
    ax.legend(fontsize=8)

    cbar = plt.colorbar(scatter, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label('Chunk index', fontsize=8)

fig.suptitle(
    'Influence Score Agreement: Physical Deletion vs Batched Mask\n'
    'Divergence from diagonal = RoPE-shift artifact in physical deletion',
    fontsize=12
)
plt.tight_layout()
plt.savefig('comparison_influence_scatter.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_influence_scatter.png')

In [ ]:
# ── Plot 5: Per-chunk influence side-by-side bar chart ────────────────────
n = len(full_del)
fig, axes = plt.subplots(n, 1, figsize=(14, 4 * n))
if n == 1:
    axes = [axes]

for ax, res_d, res_m, ex_d in zip(axes, full_del, full_mask, examples_d):
    scores_d = [c['influence_score'] for c in res_d['chunk_influences']]
    scores_m = [c['influence_score'] for c in res_m['chunk_influences']]
    m = len(scores_d)
    x = np.arange(m)
    w = 0.4

    ax.bar(x - w/2, scores_d, width=w, color='coral',     alpha=0.85, label='Physical deletion')
    ax.bar(x + w/2, scores_m, width=w, color='steelblue', alpha=0.85, label='Batched mask')
    ax.set_xticks(x)
    ax.set_xticklabels([f'C{i+1}' for i in range(m)], fontsize=8)
    ax.set_xlabel('Chunk')
    ax.set_ylabel('Influence score')

    pt   = ex_d['prompt_tokens']
    pwup_d = res_d['pwup']
    pwup_m = res_m['pwup']
    ax.set_title(
        f'{pt} tokens\n'
        f'Deletion  EUCR@10%={res_d["eucr"].get(0.1,0):.3f}  '
        f'PWUP B={pwup_d["B"]:.2f} M={pwup_d["M"]:.2f} E={pwup_d["E"]:.2f}\n'
        f'Mask      EUCR@10%={res_m["eucr"].get(0.1,0):.3f}  '
        f'PWUP B={pwup_m["B"]:.2f} M={pwup_m["M"]:.2f} E={pwup_m["E"]:.2f}',
        fontsize=10
    )
    ax.legend(fontsize=9)

fig.suptitle('Per-Chunk Influence Scores — Physical Deletion vs Batched Mask', fontsize=13)
plt.tight_layout()
plt.savefig('comparison_influence_bars.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: comparison_influence_bars.png')

In [ ]:
# ── (Optional) VRAM vs batch-size sweep ───────────────────────────────────
# Re-runs the mask ablation with several batch sizes on the first prompt only
# to characterise the speed/VRAM tradeoff curve.
# Uncomment to run (adds ~5-10 min).

# BATCH_SIZES = [1, 2, 4, 8, 0]   # 0 = all at once
# first_prompt_file = 'test_prompts_first.jsonl'
# with open(first_prompt_file, 'w') as f:
#     f.write(json.dumps(prompts[0]) + '\n')
#
# sweep_results = []
# for bs in BATCH_SIZES:
#     out_dir = f'/content/context-utilization/results_sweep_bs{bs}'
#     os.makedirs(out_dir, exist_ok=True)
#     cmd = [sys.executable, 'attention_mask_ablation.py',
#            '--model',               MODEL_ID,
#            '--input_file',          first_prompt_file,
#            '--chunk_size',          str(CHUNK_SIZE),
#            '--max_new_tokens',      str(MAX_NEW_TOKENS),
#            '--ablation_batch_size', str(bs),
#            '--dtype',               DTYPE,
#            '--output_dir',          out_dir]
#     print(f'batch_size={bs} ...', end=' ')
#     subprocess.run(cmd, check=True, capture_output=True)
#     with open(os.path.join(out_dir, 'profiling_summary.json')) as fp:
#         p = json.load(fp)
#     ex = p['examples'][0]
#     effective_bs = ex['ablation_batch_size']
#     print(f'time={ex["total_wall_time_s"]:.1f}s  vram={ex["max_peak_vram_mb"]:.0f}MB')
#     sweep_results.append({'batch_size': effective_bs,
#                           'total_time': ex['total_wall_time_s'],
#                           'vram_mb':    ex['max_peak_vram_mb']})
#
# fig, ax1 = plt.subplots(figsize=(8, 4))
# ax2 = ax1.twinx()
# xs  = [str(r['batch_size']) if r['batch_size'] else 'all' for r in sweep_results]
# ax1.plot(xs, [r['total_time'] for r in sweep_results], 'o-', color='steelblue', label='Wall time (s)')
# ax2.plot(xs, [r['vram_mb']    for r in sweep_results], 's--', color='coral',     label='Peak VRAM (MB)')
# ax1.set_xlabel('Ablation batch size')
# ax1.set_ylabel('Wall time (s)', color='steelblue')
# ax2.set_ylabel('Peak VRAM (MB)', color='coral')
# ax1.set_title('Speed / VRAM tradeoff vs batch size')
# lines = ax1.get_lines() + ax2.get_lines()
# ax1.legend(lines, [l.get_label() for l in lines], loc='upper right')
# plt.tight_layout()
# plt.savefig('sweep_batch_size.png', dpi=150, bbox_inches='tight')
# plt.show()
print('Batch-size sweep cell ready — uncomment to run.')